# Validaciones Fiscales SAT

## Deducibilidad de Primas, Siniestros y Retenciones ISR

El Servicio de Administración Tributaria (SAT) regula los aspectos fiscales de las operaciones de seguros. Este notebook demuestra las validaciones necesarias para cumplir con la normativa fiscal mexicana.

### Contenido
1. Marco fiscal de seguros en México
2. Validación de primas (deducibilidad)
3. Validación de siniestros (requisitos fiscales)
4. Cálculo de retenciones ISR
5. Casos prácticos de cumplimiento
6. Reportes fiscales automatizados

In [ ]:
# Imports necesarios
from decimal import Decimal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, date

# Imports de validaciones SAT
from mexican_insurance.regulatorio.validaciones_sat import (
    ValidadorPrimas,
    ValidadorSiniestros,
    ValidadorRetenciones
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports completados exitosamente")

## 1. Marco Fiscal de Seguros en México

### Aspectos Clave:

**Primas de Seguros (Art. 25 LISR)**
- Deducibles para personas morales (empresas)
- Límites para seguros de vida del personal
- Requisitos: CFDI, aseguradora autorizada, relación con actividad

**Indemnizaciones (Art. 93 LISR)**
- Generalmente exentas para el asegurado
- Excepciones: seguros con componente de ahorro/inversión

**Retenciones**
- ISR sobre rendimientos de seguros con ahorro
- IVA exento en primas de seguros

**Documentación Requerida**
- CFDI 3.3/4.0 con complemento de seguros
- Constancias de retenciones
- Declaraciones informativas

## 2. Validación de Primas (Deducibilidad)

Validemos si las primas pagadas son deducibles fiscalmente.

In [ ]:
# Casos de primas a validar
primas_validar = [
    {
        'id': 'P001',
        'tipo_seguro': 'GMM',
        'prima_anual': Decimal('120000'),
        'asegurado': 'Empleado',
        'cfdi_emitido': True,
        'aseguradora_autorizada': True,
        'relacionado_actividad': True,
        'beneficiario': 'Empleado y familiares'
    },
    {
        'id': 'P002',
        'tipo_seguro': 'Vida',
        'prima_anual': Decimal('50000'),
        'asegurado': 'Empleado',
        'cfdi_emitido': True,
        'aseguradora_autorizada': True,
        'relacionado_actividad': True,
        'beneficiario': 'Empresa'
    },
    {
        'id': 'P003',
        'tipo_seguro': 'Autos',
        'prima_anual': Decimal('35000'),
        'asegurado': 'Vehículo empresa',
        'cfdi_emitido': True,
        'aseguradora_autorizada': True,
        'relacionado_actividad': True,
        'beneficiario': 'Empresa'
    },
    {
        'id': 'P004',
        'tipo_seguro': 'Vida',
        'prima_anual': Decimal('80000'),
        'asegurado': 'Socio',
        'cfdi_emitido': False,  # Sin CFDI
        'aseguradora_autorizada': True,
        'relacionado_actividad': False,  # Personal del socio
        'beneficiario': 'Familiares del socio'
    },
]

print("📋 VALIDACIÓN DE DEDUCIBILIDAD DE PRIMAS")
print("="*90)

validador_primas = ValidadorPrimas()
resultados_primas = []

for prima in primas_validar:
    resultado = validador_primas.validar(
        tipo_seguro=prima['tipo_seguro'],
        prima=prima['prima_anual'],
        cfdi_emitido=prima['cfdi_emitido'],
        aseguradora_autorizada=prima['aseguradora_autorizada'],
        relacionado_actividad=prima['relacionado_actividad']
    )
    
    resultados_primas.append({
        'id': prima['id'],
        'tipo': prima['tipo_seguro'],
        'prima': prima['prima_anual'],
        'deducible': resultado['deducible'],
        'monto_deducible': resultado['monto_deducible'],
        'observaciones': resultado['observaciones']
    })
    
    print(f"\nPóliza {prima['id']} - {prima['tipo_seguro']}:")
    print(f"  Prima anual: ${prima['prima_anual']:,.0f}")
    print(f"  Deducible: {'✅ SÍ' if resultado['deducible'] else '❌ NO'}")
    print(f"  Monto deducible: ${resultado['monto_deducible']:,.0f}")
    if not resultado['deducible']:
        print(f"  ⚠️ Observaciones: {resultado['observaciones']}")

# Resumen
total_primas = sum([p['prima_anual'] for p in primas_validar])
total_deducible = sum([r['monto_deducible'] for r in resultados_primas])
total_no_deducible = total_primas - total_deducible

print(f"\n{'='*90}")
print(f"RESUMEN:")
print(f"  Total primas pagadas:    ${total_primas:>15,.0f}")
print(f"  Total deducible:         ${total_deducible:>15,.0f}")
print(f"  Total NO deducible:      ${total_no_deducible:>15,.0f}")
print(f"  % Deducible:             {(total_deducible/total_primas*100):>15.1f}%")
print(f"{'='*90}")

In [ ]:
# Visualización de deducibilidad
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Deducible vs No Deducible
labels_ded = ['Deducible', 'No Deducible']
valores_ded = [float(total_deducible), float(total_no_deducible)]
colors_ded = ['#2ecc71', '#e74c3c']

wedges, texts, autotexts = ax1.pie(valores_ded, labels=labels_ded, autopct='%1.1f%%',
                                     startangle=90, colors=colors_ded, 
                                     textprops={'fontsize': 12, 'fontweight': 'bold'})
ax1.set_title('Deducibilidad de Primas Totales', fontsize=13, fontweight='bold')

# Gráfica 2: Por póliza
ids = [r['id'] for r in resultados_primas]
primas_vals = [float(r['prima']) for r in resultados_primas]
deducibles_vals = [float(r['monto_deducible']) for r in resultados_primas]

x = np.arange(len(ids))
width = 0.35

bars1 = ax2.bar(x - width/2, primas_vals, width, label='Prima Total', 
                color='#3498db', alpha=0.7, edgecolor='black')
bars2 = ax2.bar(x + width/2, deducibles_vals, width, label='Deducible', 
                color='#2ecc71', alpha=0.7, edgecolor='black')

ax2.set_ylabel('Monto (MXN)', fontsize=12)
ax2.set_title('Deducibilidad por Póliza', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(ids)
ax2.legend(fontsize=11)
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Agregar valores
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'${height/1000:.0f}K', ha='center', va='bottom', 
                fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Validación de Siniestros (Requisitos Fiscales)

Validemos que los pagos de siniestros cumplen con los requisitos fiscales.

In [ ]:
# Casos de siniestros a validar
siniestros_validar = [
    {
        'id': 'S001',
        'tipo': 'GMM',
        'monto': Decimal('250000'),
        'tiene_cfdi': True,
        'tiene_documentacion_soporte': True,
        'beneficiario_identificado': True,
        'retenciones_aplicadas': False,  # GMM no genera retención
        'componente_ahorro': False
    },
    {
        'id': 'S002',
        'tipo': 'Vida',
        'monto': Decimal('1000000'),
        'tiene_cfdi': True,
        'tiene_documentacion_soporte': True,
        'beneficiario_identificado': True,
        'retenciones_aplicadas': False,  # Indemnización por muerte exenta
        'componente_ahorro': False
    },
    {
        'id': 'S003',
        'tipo': 'Dotal (Supervivencia)',
        'monto': Decimal('500000'),
        'tiene_cfdi': True,
        'tiene_documentacion_soporte': True,
        'beneficiario_identificado': True,
        'retenciones_aplicadas': True,  # Tiene componente de ahorro
        'componente_ahorro': True,
        'primas_pagadas': Decimal('400000'),  # Para calcular ganancia
    },
    {
        'id': 'S004',
        'tipo': 'Autos',
        'monto': Decimal('180000'),
        'tiene_cfdi': False,  # Sin CFDI
        'tiene_documentacion_soporte': True,
        'beneficiario_identificado': True,
        'retenciones_aplicadas': False,
        'componente_ahorro': False
    },
]

print("📋 VALIDACIÓN FISCAL DE SINIESTROS")
print("="*90)

validador_siniestros = ValidadorSiniestros()
resultados_siniestros = []

for siniestro in siniestros_validar:
    resultado = validador_siniestros.validar(
        tipo_seguro=siniestro['tipo'],
        monto=siniestro['monto'],
        tiene_cfdi=siniestro['tiene_cfdi'],
        tiene_documentacion=siniestro['tiene_documentacion_soporte'],
        beneficiario_identificado=siniestro['beneficiario_identificado'],
        componente_ahorro=siniestro['componente_ahorro']
    )
    
    resultados_siniestros.append({
        'id': siniestro['id'],
        'tipo': siniestro['tipo'],
        'monto': siniestro['monto'],
        'cumple': resultado['cumple'],
        'requiere_retencion': resultado['requiere_retencion'],
        'exento': resultado['exento'],
        'observaciones': resultado['observaciones']
    })
    
    print(f"\nSiniestro {siniestro['id']} - {siniestro['tipo']}:")
    print(f"  Monto: ${siniestro['monto']:,.0f}")
    print(f"  Cumple requisitos: {'✅ SÍ' if resultado['cumple'] else '❌ NO'}")
    print(f"  Exento de ISR: {'✅ SÍ' if resultado['exento'] else '❌ NO'}")
    print(f"  Requiere retención: {'⚠️ SÍ' if resultado['requiere_retencion'] else '✅ NO'}")
    if not resultado['cumple']:
        print(f"  ⚠️ Observaciones: {resultado['observaciones']}")

# Resumen
total_siniestros = sum([s['monto'] for s in siniestros_validar])
siniestros_cumplen = sum([r['monto'] for r in resultados_siniestros if r['cumple']])
siniestros_no_cumplen = total_siniestros - siniestros_cumplen

print(f"\n{'='*90}")
print(f"RESUMEN:")
print(f"  Total siniestros pagados:  ${total_siniestros:>15,.0f}")
print(f"  Cumplen requisitos:        ${siniestros_cumplen:>15,.0f}")
print(f"  NO cumplen requisitos:     ${siniestros_no_cumplen:>15,.0f}")
print(f"  % Cumplimiento:            {(siniestros_cumplen/total_siniestros*100):>15.1f}%")
print(f"{'='*90}")

In [ ]:
# Visualización de cumplimiento de siniestros
fig, ax = plt.subplots(figsize=(12, 6))

# Análisis por siniestro
ids_sin = [r['id'] for r in resultados_siniestros]
montos_sin = [float(r['monto']) for r in resultados_siniestros]
colors_sin = ['#2ecc71' if r['cumple'] else '#e74c3c' for r in resultados_siniestros]

bars = ax.bar(ids_sin, montos_sin, color=colors_sin, alpha=0.7, 
              edgecolor='black', linewidth=2)

ax.set_ylabel('Monto del Siniestro (MXN)', fontsize=12)
ax.set_title('Validación Fiscal de Siniestros\nVerde: Cumple | Rojo: No Cumple', 
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Agregar valores y etiquetas
for i, (bar, resultado) in enumerate(zip(bars, resultados_siniestros)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${height/1e6:.2f}M\n{resultado["tipo"]}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Indicador de retención
    if resultado['requiere_retencion']:
        ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
               '⚠️\nRetención', ha='center', va='center', 
               fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

## 4. Cálculo de Retenciones ISR

Calculemos las retenciones de ISR para seguros con componente de ahorro/inversión.

In [ ]:
# Casos de retenciones a calcular
casos_retenciones = [
    {
        'id': 'R001',
        'tipo': 'Dotal - Supervivencia',
        'suma_asegurada': Decimal('500000'),
        'primas_pagadas': Decimal('400000'),
        'periodo_años': 10,
        'tasa_retencion': Decimal('0.20'),  # 20% sobre rendimientos
    },
    {
        'id': 'R002',
        'tipo': 'Vida con ahorro',
        'suma_asegurada': Decimal('300000'),
        'primas_pagadas': Decimal('280000'),
        'periodo_años': 15,
        'tasa_retencion': Decimal('0.20'),
    },
    {
        'id': 'R003',
        'tipo': 'Educacional',
        'suma_asegurada': Decimal('800000'),
        'primas_pagadas': Decimal('650000'),
        'periodo_años': 12,
        'tasa_retencion': Decimal('0.20'),
    },
]

print("📋 CÁLCULO DE RETENCIONES ISR")
print("="*90)

validador_retenciones = ValidadorRetenciones()
resultados_retenciones = []

for caso in casos_retenciones:
    resultado = validador_retenciones.calcular(
        suma_asegurada=caso['suma_asegurada'],
        primas_pagadas=caso['primas_pagadas'],
        tasa_retencion=caso['tasa_retencion']
    )
    
    resultados_retenciones.append({
        'id': caso['id'],
        'tipo': caso['tipo'],
        'suma_asegurada': caso['suma_asegurada'],
        'primas_pagadas': caso['primas_pagadas'],
        'rendimiento': resultado['rendimiento'],
        'retencion_isr': resultado['retencion_isr'],
        'monto_neto': resultado['monto_neto']
    })
    
    print(f"\nCaso {caso['id']} - {caso['tipo']}:")
    print(f"  Suma Asegurada:          ${caso['suma_asegurada']:>12,.0f}")
    print(f"  Primas Pagadas:          ${caso['primas_pagadas']:>12,.0f}")
    print(f"  Rendimiento (ganancia):  ${resultado['rendimiento']:>12,.0f}")
    print(f"  Retención ISR (20%):     ${resultado['retencion_isr']:>12,.0f}")
    print(f"  Monto Neto a Pagar:      ${resultado['monto_neto']:>12,.0f}")
    
    tasa_efectiva = (resultado['rendimiento'] / caso['primas_pagadas'] * 100)
    rendimiento_anual = (pow(float(caso['suma_asegurada'] / caso['primas_pagadas']), 
                            1/caso['periodo_años']) - 1) * 100
    print(f"  Rendimiento total:       {tasa_efectiva:>12.2f}%")
    print(f"  Rendimiento anualizado:  {rendimiento_anual:>12.2f}%")

# Resumen total
total_suma = sum([r['suma_asegurada'] for r in resultados_retenciones])
total_primas = sum([r['primas_pagadas'] for r in resultados_retenciones])
total_rendimiento = sum([r['rendimiento'] for r in resultados_retenciones])
total_retencion = sum([r['retencion_isr'] for r in resultados_retenciones])
total_neto = sum([r['monto_neto'] for r in resultados_retenciones])

print(f"\n{'='*90}")
print(f"TOTALES:")
print(f"  Suma Asegurada Total:    ${total_suma:>15,.0f}")
print(f"  Primas Pagadas Total:    ${total_primas:>15,.0f}")
print(f"  Rendimiento Total:       ${total_rendimiento:>15,.0f}")
print(f"  Retención ISR Total:     ${total_retencion:>15,.0f}")
print(f"  Monto Neto Total:        ${total_neto:>15,.0f}")
print(f"{'='*90}")

In [ ]:
# Visualización de retenciones
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Composición del pago
ids_ret = [r['id'] for r in resultados_retenciones]
primas = [float(r['primas_pagadas']) for r in resultados_retenciones]
rendimientos = [float(r['rendimiento']) for r in resultados_retenciones]
retenciones = [float(r['retencion_isr']) for r in resultados_retenciones]

x = np.arange(len(ids_ret))
width = 0.6

# Barras apiladas
p1 = ax1.bar(x, primas, width, label='Primas Pagadas', color='#3498db', alpha=0.8)
p2 = ax1.bar(x, rendimientos, width, bottom=primas, 
             label='Rendimiento (antes ISR)', color='#2ecc71', alpha=0.8)

# Marcar retención
for i, ret in enumerate(retenciones):
    altura_base = primas[i] + rendimientos[i] - ret
    ax1.bar(i, ret, width, bottom=altura_base, 
            color='#e74c3c', alpha=0.8, hatch='///')

ax1.set_ylabel('Monto (MXN)', fontsize=12)
ax1.set_title('Composición de Pagos con Retención ISR', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(ids_ret)
ax1.legend(['Primas Pagadas', 'Rendimiento Neto', 'Retención ISR'], fontsize=11)
ax1.grid(axis='y', alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))

# Gráfica 2: Retenciones por caso
bars = ax2.barh(ids_ret, retenciones, color='#e74c3c', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Retención ISR (MXN)', fontsize=12)
ax2.set_title('Retenciones ISR por Caso', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))

# Agregar valores
for i, (bar, val) in enumerate(zip(bars, retenciones)):
    width_val = bar.get_width()
    ax2.text(width_val, i, f'  ${val:,.0f}', va='center', 
             fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Casos Prácticos de Cumplimiento

Analicemos casos completos de cumplimiento fiscal.

In [ ]:
# Caso 1: Empresa con prestaciones superiores
print("📊 CASO PRÁCTICO 1: EMPRESA CON PRESTACIONES SUPERIORES")
print("="*90)

prestaciones_empresa = {
    'salario_empleado_mensual': Decimal('50000'),
    'gmm_anual': Decimal('120000'),
    'vida_anual': Decimal('30000'),
    'prevision_social_anual': Decimal('150000'),  # Total otras prestaciones
}

# Límite de exención: 1 Salario Mínimo x 365 días
uma_diario = Decimal('108.57')  # 2024
limite_exento = uma_diario * 365

total_prestaciones = (prestaciones_empresa['gmm_anual'] + 
                      prestaciones_empresa['vida_anual'] + 
                      prestaciones_empresa['prevision_social_anual'])

excedente_gravado = max(Decimal('0'), total_prestaciones - limite_exento)

print(f"\nAnálisis de Prestaciones:")
print(f"  Salario Mensual:         ${prestaciones_empresa['salario_empleado_mensual']:>12,.0f}")
print(f"  GMM Anual:               ${prestaciones_empresa['gmm_anual']:>12,.0f}")
print(f"  Vida Anual:              ${prestaciones_empresa['vida_anual']:>12,.0f}")
print(f"  Otras Prestaciones:      ${prestaciones_empresa['prevision_social_anual']:>12,.0f}")
print(f"  {'─'*60}")
print(f"  Total Prestaciones:      ${total_prestaciones:>12,.0f}")
print(f"  Límite Exento (UMA):     ${limite_exento:>12,.0f}")
print(f"  Excedente Gravado:       ${excedente_gravado:>12,.0f}")

if excedente_gravado > 0:
    print(f"\n⚠️ ACCIÓN REQUERIDA:")
    print(f"   - El excedente de ${excedente_gravado:,.0f} debe integrarse al salario")
    print(f"   - Se debe calcular ISR e IMSS sobre el excedente")
    print(f"   - Considerar reestructurar prestaciones")
else:
    print(f"\n✅ Las prestaciones están dentro del límite exento")

In [ ]:
# Caso 2: Aseguradora - Declaración Informativa
print("\n📊 CASO PRÁCTICO 2: DECLARACIÓN INFORMATIVA DE ASEGURADORA")
print("="*90)

# Cargar datos de ejemplo
cartera_df = pd.read_csv('../examples/data/cartera_ejemplo.csv')
siniestros_df = pd.read_csv('../examples/data/siniestros_ejemplo.csv')

# Análisis de primas por tipo
primas_por_tipo = cartera_df.groupby('producto').agg({
    'prima_mensual': 'sum',
    'poliza_id': 'count'
}).reset_index()
primas_por_tipo.columns = ['Producto', 'Prima_Mensual', 'Num_Polizas']
primas_por_tipo['Prima_Anual'] = primas_por_tipo['Prima_Mensual'] * 12

print(f"\nDeclaración Informativa de Primas 2024:")
print(primas_por_tipo.to_string(index=False))

total_primas_anual = primas_por_tipo['Prima_Anual'].sum()
total_polizas = primas_por_tipo['Num_Polizas'].sum()

print(f"\n{'─'*70}")
print(f"TOTALES:")
print(f"  Total Pólizas:           {total_polizas:>12,}")
print(f"  Prima Anual Total:       ${total_primas_anual:>12,.0f}")

# Análisis de siniestros
print(f"\n\nDeclaración Informativa de Siniestros 2024:")
total_siniestros_pagados = siniestros_df['monto_pagado'].sum()
num_siniestros = len(siniestros_df)

print(f"  Total Siniestros Pagados: {num_siniestros:>12,}")
print(f"  Monto Total Pagado:       ${total_siniestros_pagados:>12,.0f}")

# Siniestros que requieren retención
siniestros_con_ahorro = siniestros_df[siniestros_df['tipo'].str.contains('Dotal', na=False)]
if len(siniestros_con_ahorro) > 0:
    print(f"\n⚠️ Siniestros con retención ISR: {len(siniestros_con_ahorro)}")
    print(f"   Monto sujeto a retención: ${siniestros_con_ahorro['monto_pagado'].sum():,.0f}")

print(f"\n✅ Información lista para declaración informativa")

## 6. Reportes Fiscales Automatizados

Generemos reportes automatizados para cumplimiento fiscal.

In [ ]:
# Reporte consolidado de cumplimiento fiscal
print("📊 REPORTE CONSOLIDADO DE CUMPLIMIENTO FISCAL")
print("="*90)
print(f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Período: Ejercicio 2024")
print("="*90)

# Sección 1: Primas
print(f"\n1. ANÁLISIS DE PRIMAS")
print(f"   {'─'*70}")
print(f"   Total primas emitidas:        ${total_primas:>15,.0f}")
print(f"   Deducibles:                   ${total_deducible:>15,.0f}  ({total_deducible/total_primas*100:.1f}%)")
print(f"   No deducibles:                ${total_no_deducible:>15,.0f}  ({total_no_deducible/total_primas*100:.1f}%)")

# Sección 2: Siniestros
print(f"\n2. ANÁLISIS DE SINIESTROS")
print(f"   {'─'*70}")
print(f"   Total siniestros pagados:     ${total_siniestros:>15,.0f}")
print(f"   Cumplen requisitos fiscales:  ${siniestros_cumplen:>15,.0f}  ({siniestros_cumplen/total_siniestros*100:.1f}%)")
print(f"   No cumplen requisitos:        ${siniestros_no_cumplen:>15,.0f}  ({siniestros_no_cumplen/total_siniestros*100:.1f}%)")

# Sección 3: Retenciones
print(f"\n3. RETENCIONES ISR")
print(f"   {'─'*70}")
print(f"   Total rendimientos generados: ${total_rendimiento:>15,.0f}")
print(f"   Total retenciones ISR:        ${total_retencion:>15,.0f}")
print(f"   Tasa efectiva:                {total_retencion/total_rendimiento*100:>15.2f}%")

# Sección 4: Alertas
print(f"\n4. ALERTAS Y ACCIONES REQUERIDAS")
print(f"   {'─'*70}")
alertas = []

if total_no_deducible > 0:
    alertas.append(f"⚠️  ${total_no_deducible:,.0f} en primas no deducibles - Verificar documentación")

if siniestros_no_cumplen > 0:
    alertas.append(f"⚠️  ${siniestros_no_cumplen:,.0f} en siniestros sin CFDI - Regularizar")

if total_retencion > 0:
    alertas.append(f"💰 ${total_retencion:,.0f} en retenciones pendientes de enterar")

if len(alertas) == 0:
    print("   ✅ No hay alertas - Cumplimiento al 100%")
else:
    for alerta in alertas:
        print(f"   {alerta}")

# Sección 5: Próximas obligaciones
print(f"\n5. CALENDARIO DE OBLIGACIONES FISCALES 2024")
print(f"   {'─'*70}")
print(f"   📅 Enero 17, 2024:  Pago provisional ISR (Diciembre 2023)")
print(f"   📅 Marzo 31, 2024:  Declaración anual 2023")
print(f"   📅 Abril 30, 2024:  Declaración informativa anual")
print(f"   📅 Mensual:         Declaraciones provisionales")

print(f"\n{'='*90}")
print(f"Reporte generado exitosamente")
print(f"{'='*90}")

In [ ]:
# Dashboard visual de cumplimiento fiscal
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Semáforo de cumplimiento
ax1 = fig.add_subplot(gs[0, 0])
metricas_cumplimiento = [
    ('Primas\nDeducibles', float(total_deducible/total_primas*100)),
    ('Siniestros\nCumplen', float(siniestros_cumplen/total_siniestros*100)),
    ('CFDIs\nEmitidos', 90.0),  # Simulado
]

categorias = [m[0] for m in metricas_cumplimiento]
valores = [m[1] for m in metricas_cumplimiento]
colors = ['#2ecc71' if v >= 95 else '#f39c12' if v >= 80 else '#e74c3c' for v in valores]

bars = ax1.barh(categorias, valores, color=colors, alpha=0.7, edgecolor='black')
ax1.set_xlabel('% Cumplimiento', fontsize=11)
ax1.set_title('Semáforo de Cumplimiento Fiscal', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 100)
ax1.axvline(x=80, color='orange', linestyle='--', alpha=0.5)
ax1.axvline(x=95, color='green', linestyle='--', alpha=0.5)
ax1.grid(axis='x', alpha=0.3)

for i, (bar, val) in enumerate(zip(bars, valores)):
    ax1.text(val, i, f'  {val:.1f}%', va='center', fontsize=10, fontweight='bold')

# 2. Deducibilidad de primas
ax2 = fig.add_subplot(gs[0, 1])
wedges, texts, autotexts = ax2.pie(
    [float(total_deducible), float(total_no_deducible)],
    labels=['Deducible', 'No Deducible'],
    autopct='%1.1f%%', startangle=90,
    colors=['#2ecc71', '#e74c3c'],
    textprops={'fontsize': 10, 'fontweight': 'bold'}
)
ax2.set_title('Deducibilidad de Primas', fontsize=12, fontweight='bold')

# 3. Retenciones ISR
ax3 = fig.add_subplot(gs[0, 2])
valores_ret = [float(total_rendimiento - total_retencion), float(total_retencion)]
labels_ret = ['Rendimiento Neto', 'Retención ISR']
wedges, texts, autotexts = ax3.pie(
    valores_ret, labels=labels_ret,
    autopct='%1.1f%%', startangle=90,
    colors=['#2ecc71', '#e74c3c'],
    textprops={'fontsize': 10, 'fontweight': 'bold'}
)
ax3.set_title('Distribución de Rendimientos', fontsize=12, fontweight='bold')

# 4. Timeline de obligaciones
ax4 = fig.add_subplot(gs[1, :])
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
obligaciones_mes = [1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2]  # Simulado

bars = ax4.bar(meses, obligaciones_mes, color='#3498db', alpha=0.7, edgecolor='black')
ax4.set_ylabel('Número de Obligaciones', fontsize=11)
ax4.set_title('Calendario de Obligaciones Fiscales 2024', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# Marcar meses importantes
meses_importantes = [2, 3]  # Marzo, Abril
for mes in meses_importantes:
    bars[mes].set_color('#e74c3c')
    bars[mes].set_alpha(0.7)

for bar, val in zip(bars, obligaciones_mes):
    if val > 0:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(val)}', ha='center', va='bottom', 
                fontsize=10, fontweight='bold')

plt.suptitle('Dashboard de Cumplimiento Fiscal SAT', fontsize=16, fontweight='bold', y=0.98)
plt.show()

print("\n✅ Dashboard fiscal generado exitosamente")

## Conclusiones

En este notebook aprendimos:

1. **Validación de Primas**:
   - Requisitos de deducibilidad (CFDI, relación con actividad)
   - Límites para seguros de vida del personal
   - Documentación necesaria

2. **Validación de Siniestros**:
   - Requisitos fiscales para indemnizaciones
   - Exenciones aplicables
   - Siniestros con componente de ahorro

3. **Retenciones ISR**:
   - Cálculo sobre rendimientos de seguros con ahorro
   - Tasa de retención 20%
   - Obligaciones de entero

4. **Casos Prácticos**:
   - Prestaciones superiores y límites de exención
   - Declaraciones informativas
   - Calendario de obligaciones

### Puntos Clave de Cumplimiento:
- ✅ Emitir CFDI por todas las operaciones
- ✅ Validar deducibilidad de primas antes de deducir
- ✅ Aplicar retenciones cuando corresponda
- ✅ Presentar declaraciones informativas
- ✅ Mantener documentación soporte

### Próximos Pasos

Continúa con:
- **06_reportes_cnsf.ipynb**: Generación automatizada de reportes regulatorios
- **07_casos_practicos_completos.ipynb**: Workflows end-to-end integrando todo